# Experiment 1 — Place Cell Characterisation

This notebook computes **biological plausibility metrics** for the Visual Place Cell (VPCE) ensemble across a sweep of K values and two environments. The results form the context-setting baseline for the main experimental contribution.

---

## Purpose

Establish that the revised clustering pipeline produces place cells with statistics that are consistent with biological place cell recordings: spatially compact fields, moderate population coverage, sparse and informative codes.

## Environments
- **LM8** — base landmark maze (data available)
- **LMO8** (`lm8_o6`) — open landmark variant (data collected separately)

## K sweep
`K ∈ {50, 100, 250, 500, 750}`

## Metrics
| Metric | Symbol | Description |
|---|---|---|
| Field size | $A_F$ | Mean spatial area where a cell fires above $\tau = 0.20$ of its peak |
| Coverage | $C$ | Fraction of positions covered by ≥ 1 active cell |
| Lifetime sparseness | $a_L$ | How selectively a single cell responds across positions |
| Population sparseness | $a_P$ | How few cells are active at any given position |
| Participation ratio | PR | Effective dimensionality of the population code |
| Spatial information | $I$ | Skaggs bits conveyed about location per unit activity |

---
## Metric Definitions

### Field Size  ($A_F$, threshold $\tau = 0.20$)

For each cell $i$ the **active set** is the collection of positions where its unnormalised RBF activation exceeds 20 % of that cell's peak value across the test split:

$$\mathcal{F}_i = \left\{ n : a_i(n) > \tau \cdot \max_m a_i(m) \right\}$$

Field size is estimated as the fraction of test positions in $\mathcal{F}_i$ scaled by the convex-hull area of all test positions (in m²), averaged over cells.

### Coverage  ($C$)

Fraction of test positions covered by **at least one** cell whose activation exceeds its threshold:

$$C = \frac{1}{N}\left|\bigcup_i \mathcal{F}_i\right|$$

### Lifetime Sparseness  ($a_L$, Rolls & Treves 1990)

For each cell $i$, sparseness over the $N$ test positions:

$$a_L(i) = \frac{\left(\frac{1}{N}\sum_n r_{in}\right)^2}{\frac{1}{N}\sum_n r_{in}^2}$$

Reported as the mean over cells. Values near 0 indicate sparse, selective firing; values near 1 indicate uniform firing.

### Population Sparseness  ($a_P$)

Same formula applied across the $K$ cells at each observation $n$, then averaged over positions:

$$a_P(n) = \frac{\left(\frac{1}{K}\sum_k r_{kn}\right)^2}{\frac{1}{K}\sum_k r_{kn}^2}$$

### Participation Ratio  (PR)

Effective dimensionality of the $N \times K$ activation matrix, measured via the covariance spectrum:

$$\text{PR} = \frac{\left(\operatorname{tr}\mathbf{C}\right)^2}{\operatorname{tr}\mathbf{C}^2}$$

where $\mathbf{C}$ is the $K \times K$ sample covariance matrix of column-centred activations.  PR = K means all cells contribute equally; PR ≪ K means the code is low-rank.

### Spatial Information — Skaggs Measure  ($I$)

For each cell $i$ (assuming uniform occupancy $p_n = 1/N$):

$$I_i = \frac{1}{N} \sum_n \frac{r_{in}}{\bar{r}_i} \log_2 \frac{r_{in}}{\bar{r}_i}$$

where $\bar{r}_i = \frac{1}{N}\sum_n r_{in}$.  Reported as the mean over cells (bits).

---
## Configuration

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
os.chdir('..')  # set working directory to project root

import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

plt.style.use('default')
mpl.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'savefig.facecolor': 'white',
    'text.color'       : 'black',
    'axes.labelcolor'  : 'black',
    'xtick.color'      : 'black',
    'ytick.color'      : 'black',
    'axes.edgecolor'   : 'black',
})

from realm_tools.experiment_lib.loggers import PovDataset
from realm_tools.place_cell_lib import fit_kmeans, fit_gmm

# ── Sweep parameters ────────────────────────────────────────────────────────
METHOD        = 'gmm'             # 'kmeans' | 'gmm'
K_VALUES      = [50, 100, 250, 500, 750]
N_TRIALS      = 10                # independent train/eval splits per (maze, K)
FIELD_TAU     = 0.20              # activation threshold as fraction of cell peak
REG_COVAR     = 1e-3              # GMM regularisation
TRAIN_FRAC    = 0.50              # fraction used to fit clustering

# ── Environments ────────────────────────────────────────────────────────────
# lmo8 → lm8_o6 in filesystem; add or remove entries as data becomes available
MAZES = {
    'lm8'  : {
        'data_path': 'data/vpce/collect_data/lm8',
        'maze_xml' : 'simulation/worlds/environments/vpce/lm8.xml',
    },
    'four_room' : {
        'data_path': 'data/vpce/collect_data/four_room',
        'maze_xml' : 'simulation/worlds/environments/vpce/four_room.xml',
    },
}

SAVE_DIR = 'analysis/figures/experiment_1'
os.makedirs(SAVE_DIR, exist_ok=True)

def savefig(filename):
    path = os.path.join(SAVE_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'Saved → {path}')

---
## Load Habituation Datasets

Each maze's habituation HDF5 file is loaded once.  Mazes for which data is not yet available are skipped with a warning — add the collected data to `data/vpce/collect_data/<maze>` to include them in the sweep.

In [2]:
datasets = {}

for maze, cfg in MAZES.items():
    data_path = cfg['data_path']
    h5_path   = data_path if data_path.endswith('.h5') else data_path + '.h5'
    if not os.path.exists(h5_path):
        print(f'[SKIP]  {maze}  — data not found at {h5_path}')
        continue
    ds = PovDataset.load_dataset(data_path)
    features = np.array(ds.multimodal_features)
    poses    = np.stack([ds.x, ds.y, ds.theta], axis=1)
    datasets[maze] = {'features': features, 'poses': poses}
    print(f'[OK]    {maze}  — {features.shape[0]} obs  ×  {features.shape[1]} features')

ACTIVE_MAZES = list(datasets.keys())
print(f'\nActive mazes: {ACTIVE_MAZES}')

[OK]    lm8  — 6500 obs  ×  7364 features
[OK]    four_room  — 7894 obs  ×  7364 features

Active mazes: ['lm8', 'four_room']


---
## Metric Helper Functions

All metrics operate on the **unnormalised** RBF activation matrix $\mathbf{R}$ of shape $(N_{\text{test}}, K)$, so that per-cell peaks and the field threshold are not distorted by the row-normalisation applied at runtime.

In [3]:
def raw_rbf(features, centers, radii):
    """
    Unnormalised RBF activation matrix.

    Parameters
    ----------
    features : (N, D)
    centers  : (K, D)
    radii    : (K,)

    Returns
    -------
    np.ndarray  (N, K)  — values in (0, 1]
    """
    features = np.atleast_2d(np.asarray(features, dtype=np.float64))
    f_sq    = np.sum(features ** 2, axis=1, keepdims=True)   # (N, 1)
    c_sq    = np.sum(centers  ** 2, axis=1)                   # (K,)
    fc      = features @ centers.T                             # (N, K)
    dist_sq = np.maximum(f_sq - 2 * fc + c_sq, 0.0)           # (N, K)
    return np.exp(-dist_sq / (2 * radii ** 2))                 # (N, K)


def metric_field_size(R, poses, tau=0.20):
    """
    Mean place field area (m²) estimated from the convex hull of test positions.

    Each cell's field is the set of positions where its activation exceeds
    tau * (cell peak).  Field size = (fraction of positions in field) × hull area.
    """
    xy       = poses[:, :2]
    hull_area = ConvexHull(xy).volume   # .volume == area in 2-D
    cell_max  = R.max(axis=0)           # (K,)
    active    = R > tau * cell_max      # (N, K)  boolean
    frac      = active.mean(axis=0)     # (K,)  fraction of positions active per cell
    return float((frac * hull_area).mean())


def metric_coverage(R, tau=0.20):
    """
    Fraction of test positions covered by at least one place cell
    above tau * (that cell's peak activation).
    """
    cell_max = R.max(axis=0)             # (K,)
    active   = R > tau * cell_max        # (N, K)
    return float(active.any(axis=1).mean())


def metric_lifetime_sparseness(R):
    """
    Rolls-Treves lifetime sparseness, averaged over cells.
    0 = maximally sparse (one-position firing); 1 = uniform firing.
    """
    mu  = R.mean(axis=0)          # (K,)
    mu2 = (R ** 2).mean(axis=0)   # (K,)
    with np.errstate(invalid='ignore', divide='ignore'):
        s = np.where(mu2 > 0, mu ** 2 / mu2, 0.0)
    return float(s.mean())


def metric_population_sparseness(R):
    """
    Population sparseness per observation, averaged over positions.
    0 = one cell active at a time; 1 = all cells equally active.
    """
    mu  = R.mean(axis=1)          # (N,)
    mu2 = (R ** 2).mean(axis=1)   # (N,)
    with np.errstate(invalid='ignore', divide='ignore'):
        s = np.where(mu2 > 0, mu ** 2 / mu2, 0.0)
    return float(s.mean())


def metric_participation_ratio(R):
    """
    Effective dimensionality of the activation matrix via covariance trace ratio:
        PR = tr(C)² / tr(C²)
    where C is the K×K sample covariance of (column-centred) R.
    PR = K means all eigenmodes contribute equally;
    PR ≪ K means the representation is low-rank.
    """
    A = R - R.mean(axis=0)            # centre columns  (N, K)
    C = (A.T @ A) / len(R)            # K × K covariance
    tr_C  = np.trace(C)
    tr_C2 = np.trace(C @ C)
    return float(tr_C ** 2 / tr_C2) if tr_C2 > 0 else 0.0


def metric_spatial_information(R):
    """
    Skaggs spatial information (bits) per cell, assuming uniform occupancy.
    Mean over cells with non-zero mean firing rate.

        I_i = (1/N) Σ_n (r_in / r̄_i) log₂(r_in / r̄_i)

    Zero-activity terms contribute 0 (lim x→0 of x log x = 0).
    """
    mean_r = R.mean(axis=0)                       # (K,)
    valid  = mean_r > 0
    if not valid.any():
        return 0.0
    safe_mean = np.where(valid, mean_r, 1.0)      # avoid div-by-zero for invalid cols
    ratio     = R / safe_mean                      # (N, K)
    with np.errstate(divide='ignore', invalid='ignore'):
        log_r = np.where(ratio > 0, np.log2(ratio), 0.0)
    per_cell = (ratio * log_r).mean(axis=0)        # (K,)
    return float(per_cell[valid].mean())


print('Metric functions defined.')

Metric functions defined.


---
## K-Sweep  (N_TRIALS independent trials per combination)

For each **(maze, K, trial)** combination:

1. Draw a fresh **train / eval split** using `seed = k_idx * 100 + trial` — every (K, trial) pair gets a unique but reproducible seed.
2. Fit the clustering with the same seed as `random_state`, so GMM initialisation is independently varied across trials.
3. Compute **unnormalised RBF activations** on the eval split.
4. Evaluate all six metrics and append one row to `records`.

After the loop a second aggregation step groups by `(maze, K)` and computes **mean ± std** across the `N_TRIALS` rows, retaining only the three primary reported metrics: field size, lifetime sparseness, and spatial information.

In [ ]:
records = []

for maze in ACTIVE_MAZES:
    features_all = datasets[maze]['features']
    poses_all    = datasets[maze]['poses']
    N            = len(features_all)

    for k_idx, K in enumerate(K_VALUES):
        trial_metrics = []  # accumulate per-trial dicts for the summary line

        for trial in range(N_TRIALS):
            seed = k_idx * 100 + trial                     # unique per (K, trial)
            rng  = np.random.default_rng(seed=seed)
            perm = rng.permutation(N)
            n_train   = int(N * TRAIN_FRAC)
            train_idx = perm[:n_train]
            eval_idx  = perm[n_train:]

            X_train = features_all[train_idx]
            X_eval  = features_all[eval_idx]
            P_eval  = poses_all[eval_idx]

            # Fit clustering — random_state mirrors the split seed so GMM
            # initialisation is independently varied across trials
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                if METHOD == 'kmeans':
                    centers, radii = fit_kmeans(X_train, n_clusters=K)
                elif METHOD == 'gmm':
                    centers, radii = fit_gmm(
                        X_train, n_components=K,
                        reg_covar=REG_COVAR,
                        random_state=seed,
                    )
                else:
                    raise ValueError(f'Unknown METHOD: {METHOD}')

            # Unnormalised activations on eval split
            R = raw_rbf(X_eval, centers, radii)    # (N_eval, K)

            rec = {
                'maze'                  : maze,
                'K'                     : K,
                'trial'                 : trial,
                'n_train'               : n_train,
                'n_eval'                : len(eval_idx),
                'field_size_m2'         : metric_field_size(R, P_eval, tau=FIELD_TAU),
                'coverage'              : metric_coverage(R, tau=FIELD_TAU),
                'lifetime_sparseness'   : metric_lifetime_sparseness(R),
                'population_sparseness' : metric_population_sparseness(R),
                'participation_ratio'   : metric_participation_ratio(R),
                'spatial_info_bits'     : metric_spatial_information(R),
            }
            records.append(rec)
            trial_metrics.append(rec)

            if trial == 0:
                print(
                    f'{maze:6s}  K={K:>4d}  trial={trial}  '
                    f'FS={rec["field_size_m2"]:.3f} m²  '
                    f'aL={rec["lifetime_sparseness"]:.4f}  '
                    f'I={rec["spatial_info_bits"]:.3f} bits'
                )

        # Summary line after all trials for this (maze, K)
        fs_vals = [r['field_size_m2']       for r in trial_metrics]
        al_vals = [r['lifetime_sparseness']  for r in trial_metrics]
        si_vals = [r['spatial_info_bits']    for r in trial_metrics]
        print(
            f'{maze:6s}  K={K:>4d}  [{N_TRIALS} trials]  '
            f'FS={np.mean(fs_vals):.3f}±{np.std(fs_vals):.3f} m²  '
            f'aL={np.mean(al_vals):.4f}±{np.std(al_vals):.4f}  '
            f'I={np.mean(si_vals):.3f}±{np.std(si_vals):.3f} bits'
        )

results = pd.DataFrame(records)

# ── Aggregation: mean ± std across trials for the three primary metrics ──────
REPORTED_METRICS = ['field_size_m2', 'lifetime_sparseness', 'spatial_info_bits']

agg_funcs = {col: ['mean', 'std'] for col in REPORTED_METRICS}
summary = results.groupby(['maze', 'K'], sort=False).agg(agg_funcs)
summary.columns = [f'{col}_{stat}' for col, stat in summary.columns]
summary = summary.reset_index()

print(f'\nRaw records  : {len(results)} rows  (maze × K × trial)')
print(f'Summary rows : {len(summary)} rows  (maze × K)')

lm8     K=  50  trial=0  FS=10.697 m²  aL=0.7028  I=0.340 bits


---
## Results Table

In [ ]:
# ── Summary table (mean ± std for the three reported metrics) ─────────────────
fmt_summary = {
    'field_size_m2_mean'       : '{:.4f}'.format,
    'field_size_m2_std'        : '{:.4f}'.format,
    'lifetime_sparseness_mean' : '{:.4f}'.format,
    'lifetime_sparseness_std'  : '{:.4f}'.format,
    'spatial_info_bits_mean'   : '{:.4f}'.format,
    'spatial_info_bits_std'    : '{:.4f}'.format,
}
print('── Aggregated summary (mean ± std across trials) ──')
print(summary.to_string(index=False, formatters=fmt_summary))

# ── Raw records table (all metrics, all trials) ────────────────────────────────
fmt_raw = {
    'field_size_m2'         : '{:.4f}'.format,
    'coverage'              : '{:.3f}'.format,
    'lifetime_sparseness'   : '{:.4f}'.format,
    'population_sparseness' : '{:.4f}'.format,
    'participation_ratio'   : '{:.2f}'.format,
    'spatial_info_bits'     : '{:.4f}'.format,
}
print('\n── Raw records (all metrics, all trials) ──')
raw_cols = ['maze', 'K', 'trial', 'field_size_m2', 'coverage',
            'lifetime_sparseness', 'population_sparseness',
            'participation_ratio', 'spatial_info_bits']
print(results[raw_cols].to_string(index=False, formatters=fmt_raw))

# Save both to CSV
summary_csv = os.path.join(SAVE_DIR, 'experiment_1_summary.csv')
raw_csv     = os.path.join(SAVE_DIR, 'experiment_1_raw.csv')
summary.to_csv(summary_csv, index=False)
results.to_csv(raw_csv,     index=False)
print(f'\nSaved → {summary_csv}')
print(f'Saved → {raw_csv}')

---
## Metric Plots

Each metric is plotted as a function of K, with one line per maze.  Markers are plotted at each K value in the sweep.

In [ ]:
# ── Field Size (mean ± std across trials) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for maze in ACTIVE_MAZES:
    sub = summary[summary['maze'] == maze]
    ax.errorbar(sub['K'], sub['field_size_m2_mean'], yerr=sub['field_size_m2_std'],
                marker='o', capsize=4, label=maze.upper())
ax.set_xlabel('Number of place cells  K', fontsize=13)
ax.set_ylabel(f'Mean field size  (m²,  τ = {FIELD_TAU})', fontsize=13)
ax.set_title(f'Field Size vs K — {METHOD.upper()}  (mean ± std, {N_TRIALS} trials)', fontsize=13)
ax.set_xscale('log')
ax.set_xticks(K_VALUES)
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
ax.legend(fontsize=11)
plt.tight_layout()
savefig(f'field_size_{METHOD}.png')
plt.show()

In [ ]:
# ── Coverage ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for maze in ACTIVE_MAZES:
    sub = results[results['maze'] == maze]
    ax.plot(sub['K'], sub['coverage'], marker='o', label=maze.upper())
ax.axhline(1.0, color='grey', linestyle=':', linewidth=1, label='Full coverage')
ax.set_xlabel('Number of place cells  K', fontsize=13)
ax.set_ylabel(f'Coverage  (τ = {FIELD_TAU})', fontsize=13)
ax.set_title(f'Maze Coverage vs K — {METHOD.upper()}', fontsize=13)
ax.set_xscale('log')
ax.set_xticks(K_VALUES)
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
plt.tight_layout()
savefig(f'coverage_{METHOD}.png')
plt.show()

In [ ]:
# ── Lifetime Sparseness (mean ± std) / Population Sparseness (raw mean) ───────
# Lifetime sparseness is a reported metric — plotted from summary with error bars.
# Population sparseness is kept in raw records only — plotted as plain mean line.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for maze in ACTIVE_MAZES:
    sub_s = summary[summary['maze'] == maze]
    sub_r = results[results['maze'] == maze].groupby('K')['population_sparseness'].mean().reset_index()
    axes[0].errorbar(sub_s['K'], sub_s['lifetime_sparseness_mean'],
                     yerr=sub_s['lifetime_sparseness_std'],
                     marker='o', capsize=4, label=maze.upper())
    axes[1].plot(sub_r['K'], sub_r['population_sparseness'], marker='o', label=maze.upper())

for ax, title, ylabel in [
    (axes[0], f'Lifetime Sparseness vs K  (mean ± std, {N_TRIALS} trials)', 'Lifetime sparseness  $a_L$'),
    (axes[1], 'Population Sparseness vs K  (trial mean)',                   'Population sparseness  $a_P$'),
]:
    ax.set_xlabel('Number of place cells  K', fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(f'{title} — {METHOD.upper()}', fontsize=13)
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=11)

plt.tight_layout()
savefig(f'sparseness_{METHOD}.png')
plt.show()

In [ ]:
# ── Participation Ratio ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for maze in ACTIVE_MAZES:
    sub = results[results['maze'] == maze]
    ax.plot(sub['K'], sub['participation_ratio'], marker='o', label=maze.upper())
    # Reference line: PR = K (all modes equal)
k_arr = np.array(K_VALUES, dtype=float)
ax.plot(k_arr, k_arr, color='grey', linestyle=':', linewidth=1, label='PR = K  (uniform)')
ax.set_xlabel('Number of place cells  K', fontsize=13)
ax.set_ylabel('Participation ratio  PR', fontsize=13)
ax.set_title(f'Participation Ratio vs K — {METHOD.upper()}', fontsize=13)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xticks(K_VALUES)
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
ax.legend(fontsize=11)
plt.tight_layout()
savefig(f'participation_ratio_{METHOD}.png')
plt.show()

In [ ]:
# ── Spatial Information — Skaggs (mean ± std across trials) ───────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for maze in ACTIVE_MAZES:
    sub = summary[summary['maze'] == maze]
    ax.errorbar(sub['K'], sub['spatial_info_bits_mean'], yerr=sub['spatial_info_bits_std'],
                marker='o', capsize=4, label=maze.upper())
ax.set_xlabel('Number of place cells  K', fontsize=13)
ax.set_ylabel('Spatial information  (bits)', fontsize=13)
ax.set_title(f'Skaggs Spatial Information vs K — {METHOD.upper()}  (mean ± std, {N_TRIALS} trials)', fontsize=13)
ax.set_xscale('log')
ax.set_xticks(K_VALUES)
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
ax.legend(fontsize=11)
plt.tight_layout()
savefig(f'spatial_information_{METHOD}.png')
plt.show()

---
## Summary Figure

All six metrics in a single 2 × 3 panel for inclusion in the paper.

In [ ]:
# Reported metrics use summary (mean ± std); others use per-trial mean from results.
METRIC_SPECS = [
    # (column_or_prefix, ylabel, log_y, from_summary)
    ('field_size_m2',         f'Field size  (m²,  τ={FIELD_TAU})',  False, True),
    ('coverage',              f'Coverage  (τ={FIELD_TAU})',          False, False),
    ('lifetime_sparseness',   'Lifetime sparseness  $a_L$',          False, True),
    ('population_sparseness', 'Population sparseness  $a_P$',        False, False),
    ('participation_ratio',   'Participation ratio  PR',             True,  False),
    ('spatial_info_bits',     'Spatial information  (bits)',         False, True),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, ylabel, log_y, from_summary) in zip(axes, METRIC_SPECS):
    for maze in ACTIVE_MAZES:
        if from_summary:
            sub = summary[summary['maze'] == maze]
            ax.errorbar(sub['K'], sub[f'{col}_mean'], yerr=sub[f'{col}_std'],
                        marker='o', markersize=5, capsize=3, label=maze.upper())
        else:
            sub = results[results['maze'] == maze].groupby('K')[col].mean().reset_index()
            ax.plot(sub['K'], sub[col], marker='o', markersize=5, label=maze.upper())
    ax.set_xlabel('K', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
    ax.tick_params(axis='x', labelrotation=45)
    if log_y:
        ax.set_yscale('log')
    ax.legend(fontsize=10)

fig.suptitle(
    f'Experiment 1 — Place Cell Characterisation  ({METHOD.upper()},  {N_TRIALS} trials)\n'
    f'Error bars = ± std  (reported metrics only)',
    fontsize=14, y=1.02,
)
plt.tight_layout()
savefig(f'experiment_1_summary_{METHOD}.png')
plt.show()